# Stage A2 — Exploratory Analysis

Dual-fuel PINN pipeline | Sandrine Schueller Mafra | PPGEM – UFPR

**Input:** `data/masters_data.xlsx` (read directly — this notebook has no
hard-coded values from the dissertation text, since Stage A1 already
showed several of those don't match the real data).

**What this notebook computes, straight from the data:**
- Descriptive statistics
- Per-variable distribution + kernel density estimate (KDE)
- Pearson and Spearman correlation matrices (9×9, inputs and outputs)
- Variance Inflation Factor (VIF) for the 4 input variables
- IQR-based outlier screening per variable

**Output:** nothing is written to disk by default — this stage is for
inspection. If a later stage needs any of these results, save the
relevant cell's output explicitly.


## Setup

In [ ]:
import numpy as np
import polars as pl
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy.stats import gaussian_kde, spearmanr
from pathlib import Path

print("polars ", pl.__version__)
import plotly
print("plotly ", plotly.__version__)

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "code" else Path.cwd()
RAW_PATH = PROJECT_ROOT / "data" / "masters_data.xlsx"
RAW_PATH


## 1. Load data

Reads the Excel file directly with `polars` (requires the optional
`fastexcel` dependency: `pip install polars fastexcel` if this errors
with a missing-engine message) and renames columns to short, code-
friendly names. The raw `SO_H [FSN]` column is the particulate-matter
channel and `ETA [%]` is stored as a **fraction** (0–1) in the file
despite its header — both handled explicitly below, nothing silently
converted.

In [ ]:
COLUMN_MAP = {
    "SOI [o.CA]": "SOI",
    "Lambda [-]": "lambda",
    "Sub. Rate [%]": "sub_rate",
    "Prail [bar]": "P_rail",
    "HC [g/kW.h]": "HC",
    "NOX [ppm]": "NOx",
    "CO2 [%]": "CO2",
    "SO_H [FSN]": "PM",
    "ETA [%]": "eta",
}
INPUT_COLS = ["SOI", "lambda", "sub_rate", "P_rail"]
OUTPUT_COLS = ["HC", "NOx", "CO2", "PM", "eta"]
ALL_COLS = INPUT_COLS + OUTPUT_COLS

df = pl.read_excel(RAW_PATH)
df = df.rename(COLUMN_MAP).select(ALL_COLS)

print(df.shape)
df.head()


## 2. Descriptive statistics

In [ ]:
df.describe()


## 3. Distribution & density (KDE) per variable

Histogram (density-normalised) with a Gaussian KDE overlay, one panel
per variable, computed straight from the loaded data.

In [ ]:
fig = make_subplots(rows=3, cols=3, subplot_titles=ALL_COLS)

for i, col in enumerate(ALL_COLS):
    row, c = divmod(i, 3)
    values = df[col].to_numpy()
    kde = gaussian_kde(values)
    x_grid = np.linspace(values.min(), values.max(), 200)
    density = kde(x_grid)

    fig.add_trace(
        go.Histogram(x=values, histnorm="probability density",
                     marker_color="#185FA5", opacity=0.55, showlegend=False,
                     nbinsx=12),
        row=row + 1, col=c + 1,
    )
    fig.add_trace(
        go.Scatter(x=x_grid, y=density, mode="lines",
                    line=dict(color="#993C1D", width=2), showlegend=False),
        row=row + 1, col=c + 1,
    )

fig.update_layout(height=800, width=950,
                   title_text="Distribution + KDE per variable (masters_data.xlsx)")
fig.show()


## 4. Correlation — Pearson

Computed with `numpy.corrcoef` on the raw column values (not polars'
own `.corr`, so the result doesn't depend on which correlation method a
given polars version ships) and displayed as an interactive heatmap.

In [ ]:
X = df.select(ALL_COLS).to_numpy()
pearson_mat = np.corrcoef(X, rowvar=False)

pearson_data = {"variable": ALL_COLS}
for i, c in enumerate(ALL_COLS):
    pearson_data[c] = pearson_mat[:, i]
pearson_df = pl.DataFrame(pearson_data)
pearson_df


In [ ]:
fig = px.imshow(
    pearson_mat, x=ALL_COLS, y=ALL_COLS, text_auto=".2f",
    color_continuous_scale="RdBu_r", zmin=-1, zmax=1, aspect="auto",
    title="Pearson correlation matrix",
)
fig.update_layout(width=650, height=600)
fig.show()


## 5. Correlation — Spearman

Rank-based correlation (via `scipy.stats.spearmanr`); picks up monotonic
relationships that aren't necessarily linear, which matters here since
several inputs/outputs are physically nonlinear (e.g. a U-shaped HC–λ
relationship would show up weakly in Pearson but should still show up
here if it's monotonic on each side).

In [ ]:
spearman_mat, _ = spearmanr(X)
fig = px.imshow(
    spearman_mat, x=ALL_COLS, y=ALL_COLS, text_auto=".2f",
    color_continuous_scale="RdBu_r", zmin=-1, zmax=1, aspect="auto",
    title="Spearman correlation matrix",
)
fig.update_layout(width=650, height=600)
fig.show()


## 6. Multicollinearity — Variance Inflation Factor (inputs only)

VIF is computed here with plain linear algebra (no `statsmodels`
dependency): for standardised predictors, `VIF_j` is the *j*-th diagonal
element of the inverse of their correlation matrix — algebraically the
same result `1 / (1 - R²_j)` from regressing each input on the other
three would give.

Rule of thumb: VIF < 5 is comfortable, VIF > 10 signals a real problem.

In [ ]:
Xin = df.select(INPUT_COLS).to_numpy()
Xin_std = (Xin - Xin.mean(axis=0)) / Xin.std(axis=0, ddof=1)
R = np.corrcoef(Xin_std, rowvar=False)
vif = np.diag(np.linalg.inv(R))

vif_df = pl.DataFrame({"variable": INPUT_COLS, "VIF": vif})
vif_df


In [ ]:
fig = go.Figure(go.Bar(x=INPUT_COLS, y=vif, marker_color="#3B6D11"))
fig.add_hline(y=5, line_dash="dash", line_color="#993C1D",
              annotation_text="VIF = 5 (caution threshold)")
fig.update_layout(title="Variance Inflation Factor per input", yaxis_title="VIF",
                   width=650, height=400)
fig.show()


## 7. Outlier screening (IQR method)

Standard Tukey rule: a value is flagged if it falls outside
`[Q1 − 1.5·IQR, Q3 + 1.5·IQR]`.

**Read the counts with this dataset's design in mind:** the test matrix
is one-factor-at-a-time (Sec. 3.1.1.2), so most rows repeat a shared
baseline for three of the four inputs while only one is swept. That
means a variable's own inter-quartile range can be very narrow (many
rows sit exactly at the baseline), so genuine, intentional sweep points
get flagged as "outliers" here even though they're the most informative
rows in the dataset, not data-quality problems. Treat this table as
*"far from the modal condition"*, not *"suspect measurement."*

In [ ]:
def iqr_outliers(frame: pl.DataFrame, col: str):
    q1 = frame[col].quantile(0.25, interpolation="linear")
    q3 = frame[col].quantile(0.75, interpolation="linear")
    iqr = q3 - q1
    lo, hi = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    flagged = frame.filter((pl.col(col) < lo) | (pl.col(col) > hi))
    return len(flagged), lo, hi, flagged


rows = []
for c in ALL_COLS:
    n, lo, hi, _ = iqr_outliers(df, c)
    rows.append({"variable": c, "n_outliers": n, "lower_bound": lo, "upper_bound": hi})

outlier_summary = pl.DataFrame(rows)
outlier_summary


In [ ]:
fig = go.Figure(go.Bar(x=outlier_summary["variable"], y=outlier_summary["n_outliers"],
                        marker_color="#854F0B"))
fig.update_layout(title="IQR-flagged points per variable (see note above before reacting to this)",
                   yaxis_title="count", width=650, height=400)
fig.show()


## Next

Stage A3 (normalisation + stratified train/val/test split) can reuse
`INPUT_COLS` / `OUTPUT_COLS` from this notebook. The VIF result above
(all ≈1, by design) means no input needs to be dropped or combined
before modelling; the correlation matrices are the reference point to
come back to once the physics constraints (Table 7, Sec. 3.3) are
implemented in Phase C, to check the learned constraint signs against
what the raw data already shows.